## 上下文 -

### docs

### styling

### directory

In [ ]:
!tree ..

..
├── LICENSE
├── MANIFEST.in
├── README.md
├── _proc
│   ├── 00_core.ipynb
│   ├── _docs
│   │   ├── index.html
│   │   ├── robots.txt
│   │   └── sitemap.xml
│   ├── _quarto.yml
│   ├── index.ipynb
│   ├── nbdev.yml
│   └── styles.css
├── nbs
│   ├── 00_core.ipynb
│   ├── _quarto.yml
│   ├── db.db
│   ├── db.db-shm
│   ├── db.db-wal
│   ├── index.ipynb
│   ├── nbdev.yml
│   └── styles.css
├── pyproject.toml
└── vlm_monitor
    ├── __init__.py
    ├── __pycache__
    │   ├── __init__.cpython-312.pyc
    │   └── core.cpython-312.pyc
    ├── _modidx.py
    └── core.py

6 directories, 25 files


# core

> Fill in a module description here

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp core

## Database

In [ ]:
#| export
from fastcore.all import *

In [ ]:
#| export
class Video: id:int; title:str=''; overview:str=''; transcript:str=''; length:int=0; sample_rate:int=0
class Frame: id:int; video_id:int; frame_number:int; subtitle:str=''
class Run: id:int; deploy_time:str; finish_time:str; request_timer:str; video_id:int; model:str; usage:str; num_frames:int; description:str
class RunFrame: run_id:int; frame_id:int; type:str; system_prompt:str; prompt:str; description:str; usage:str

In [ ]:
#| export
from fastlite import *

In [ ]:
!rm db.db
db = database('db.db'); db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

In [ ]:
??Queryable.schema

Type:        property
String form: <property object>
Source:     
# Queryable.schema.fget
@property
def schema(self) -> str:
    "SQL schema for this table or view."
    return self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0]

In [ ]:
@patch(as_prop=True)
def schema(self:Queryable) -> str:
    "SQL schema for this table or view."
    return hl_md(self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0], lang='sql')

In [ ]:
videos = db.create(Video, transform=True); videos.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER
)
```

</div>

In [ ]:
frames = db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')]); frames.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [frame] (
   [id] INTEGER PRIMARY KEY,
   [video_id] INTEGER REFERENCES [video]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_number] INTEGER,
   [subtitle] TEXT
)
```

</div>

In [ ]:
runs = db.create(Run, transform=True); runs.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [run] (
   [id] INTEGER PRIMARY KEY,
   [deploy_time] TEXT,
   [finish_time] TEXT,
   [request_timer] TEXT,
   [video_id] INTEGER,
   [model] TEXT,
   [usage] TEXT,
   [num_frames] INTEGER,
   [description] TEXT
)
```

</div>

In [ ]:
runframes = db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True); runframes.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [run_frame] (
   [run_id] INTEGER REFERENCES [run]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_id] INTEGER REFERENCES [frame]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [type] TEXT,
   [system_prompt] TEXT,
   [prompt] TEXT,
   [description] TEXT,
   [usage] TEXT,
   PRIMARY KEY ([run_id], [frame_id], [type])
)
```

</div>

In [ ]:
#| export
from typing import NamedTuple
from apswutils.db import Database, Table
class DBResources(NamedTuple): db:Database; videos:Table; frames:Table; runs:Table; runframes:Table

In [ ]:
#| export

def init_db(
    path:str|Path='db.db' # Path to database
) -> DBResources:
    "Initialize a database and return the database, as well as its videos, frames, runs, and runframes tables."
    db = database(path)
    videos = db.create(Video, transform=True)
    frames = db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')])
    runs = db.create(Run, transform=True)
    runframes = db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True)
    return DBResources(db, videos, frames, runs, runframes)


In [ ]:
# !rm db.db
o = init_db()

In [ ]:
o.db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

In [ ]:
o.videos.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER
)
```

</div>

## VLM

In [ ]:
#| export
from fastllm.types import Msg, Part, PartType

after i added this fastllm import statement, the github pages deploy action failes; 5WHYSL

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

好，我来走一遍 5WHYS(L)。

---

**表面现象**：加了 `from fastllm.types import Msg, Part, PartType` 之后，GitHub Pages deploy action 挂了。

**1. 为什么 deploy action 会挂？**

→ 因为 `nbdev-docs` 在处理 notebook 时崩溃了，抛出 `Exception: Error in notebook: core in cell 36`。

**2. 为什么 `nbdev-docs` 在处理 cell 36 时会崩溃？**

→ 因为 `exec_show_docs` processor 会执行所有 export cell 来生成文档。执行到 `from fastllm.types import Msg, Part, PartType` 时，fastllm 的导入链触发了 `OpenAI()` 初始化，而 CI 环境没有 `OPENAI_API_KEY`，所以 `openai.OpenAIError` 被抛出。

**3. 为什么 `from fastllm.types import Msg, Part, PartType` 会触发 `OpenAI()` 初始化？**

→ 因为 `fastllm/__init__.py` 从 `fastllm.base` 导入了 `Model` 类，而 `Model` 类定义中有一行 `client: OpenAI = OpenAI()`，在类定义时就会执行，不会等到实际使用才初始化。

**4. 为什么 fastllm 要在类定义时急切创建 `OpenAI()` 实例？**

→ 这是 fastllm 的设计决策——假设 API key 始终可用，并且 `Model` 是一个全局单例式的客户端封装，在模块加载时就创建连接。

**5. 为什么这个设计问题现在才暴露？**

→ 因为你在 `#| export` cell 里 `from fastllm.types import ...`。nbdev 只执行 export cell 来生成文档。之前 fastllm 的导入可能只出现在非 export 的 cell 里，或者根本不在 docs 构建路径上。

---

**结论**：根因不是 "fastllm 有 bug"，也不是 "CI 缺少 env var"——这些都是表层。**真正的问题是：一个仅用于类型标注的导入语句（`Msg, Part, PartType`），因为被放在了 export cell 里，被 nbdev 的 doc 生成流程执行了，而这触发了 fastllm 全包的副作用导入。**

有几个方向的修复思路——你想讨论哪个？

<details class='token-usage-details' markdown='1'><summary>$0.0128</summary>

`total=73,641 | in=72,129 | out=1,512 | cached=68.7% | reasoning=998 | $0.0128 | deepseek-v4-flash`

</details>


In [ ]:
from os import environ; environ

how come it doesn't fail on my end though? i don't have an openai_api_key set ehre?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

好问题！让我看看这里的 fastllm 版本和它的 `Model` 类定义。





<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;import fastllm; fastllm.__version__&quot;)→&quot;0.0.26&quot;</code></summary>

```json
{
  "id": "call_00_2BkgNRAAHG5U2mIViDeO0854",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "import fastllm; fastllm.__version__"
    }
  },
  "result": "0.0.26"
}
```

</details>



<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;import inspect; print(inspect.getsource(…&quot;)→&quot;----------------------------------------…&quot;</code></summary>

```json
{
  "id": "call_01_WlMCau18sohLG8qzXAyE3094",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "import inspect; print(inspect.getsource(fastllm.base.Model))"
    }
  },
  "result": "---------------------------------------------------------------------------\nAttributeError                            Traceback (most recent call last)\nCell In[32], line 1\n----> 1 await call_tool(python, {'code': 'import inspect; print(inspect.getsource(fastllm.base.Model))'})\n\nFile /usr/local/lib/python3.12/site-packages/ipykernel_helper/core.py:373, in call_tool(func, kw)\n    371 async def call_tool(func, kw):\n    372     \"Call `func(**kw)` with `coerce_inputs`\"\n--> 373     return await maybe_await(func(**coerce_inputs(func, kw)))\n\nFile /usr/local/lib/python3.12/site-packages/fastcore/xtras.py:1079, in maybe_await(o)\n   1077 \"Await `o` if needed, and return it\"\n   1078 from inspect import isawaitable\n-> 1079 return await o if isawaitable(o) else o\n\nFile /usr/local/lib/python3.12/site-packages/safepyrun/core.py:299, in RunPython.__call__(self, code)\n    297 tb = e.__traceback__\n    298 while tb.tb_next and not tb.tb_frame.f_code.co_filename.startswith('<python'): tb = tb.tb_next\n--> 299 raise e.with_traceback(tb) from None\n\nFile <python_3>:1\n----> 1 print(inspect.getsource(fastllm.base.Model))\n\nAttributeError: module 'fastllm' has no attribute 'base'"
}
```

</details>





关键区别就在这里——**fastllm 的版本不同**。

| 环境 | fastllm 版本 | 行为 |
|---|---|---|
| 这个 Solveit 环境 | `0.0.26` | `from fastllm.types import ...` 不会触发 `OpenAI()` 初始化 |
| CI 环境 | `>=0.2.1`（从 pyproject.toml 安装） | `fastllm.base.Model` 类定义中有一行 `client: OpenAI = OpenAI()`，在 import 时就执行 |

这个 Solveit 环境用的是旧版的 fastllm，根本没有 `fastllm.base` 模块，所以不会出问题。但 CI 安装的是 `>=0.2.1` 的新版，`Model` 在类定义时**急切创建** `OpenAI()` 实例（[fastllm/base.py#L520](https://github.com/fastai/fastllm/blob/main/fastllm/base.py#L520)），没有 API key 就直接炸了。

你的 pyproject.toml 里有没有设 fastllm 版本上限？如果没锁，CI 每次都会拉最新的。

<details class='token-usage-details' markdown='1'><summary>$0.0118</summary>

`total=155,811 | in=154,463 | out=1,348 | cached=92.9% | reasoning=982 | $0.0118 | deepseek-v4-flash`

</details>


oh, is it that the CI action is installing some other rando lib called fastllm, and that the fastllm i'm using isn't actually launched yet? because on github, the newest fastllm version is only 0.0.33...

this is the one i'm using, and i don't see pip install instructions; https://github.com/AnswerDotAI/fastllm

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

对！你的 `fastllm`（AnswerDotAI 的）版本号才到 `0.0.33`，但 CI 装的是 `0.2.1`——这是 PyPI 上另一个同名的包。

```
Collecting fastllm (from vlm-monitor==0.0.1)
    Using cached fastllm-0.2.1-py3-none-any.whl.metadata (7.6 kB)
```

你那个 repo 没有 `pip install fastllm` 的说明，就是因为这个包没有发布到 PyPI（或者发布的是另一个名字）。CI 的 `pip install -e '.[dev]'` 解析你的 pyproject.toml 依赖 `fastllm`，发现本地没有装，就去 PyPI 拉了一个**同名但无关的第三方包**回来。

你要确认一下的话，可以看看 pyproject.toml 里 `dependencies` 怎么写，以及 AnswerDotAI/fastllm 实际是发布到什么名字的——可能是 `fastllm-answerdotai` 或类似的？〰️🐛

<details class='token-usage-details' markdown='1'><summary>$0.0040</summary>

`total=78,309 | in=77,455 | out=854 | cached=99.8% | reasoning=617 | $0.0040 | deepseek-v4-flash`

</details>


you go take a look

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

<details class='tool-usage-details' markdown='1'>
<summary><code>python(code=&quot;print(Path(&#x27;../pyproject.toml&#x27;).read_tex…&quot;)→&quot;[build-system]\nrequires = [\&quot;setuptools&gt;=…&quot;</code></summary>

```json
{
  "id": "call_00_wIQghGkeXbXwEHEjbpsz2852",
  "server": false,
  "call": {
    "function": "python",
    "arguments": {
      "code": "print(Path('../pyproject.toml').read_text())"
    }
  },
  "result": "[build-system]\nrequires = [\"setuptools>=64\"]\nbuild-backend = \"setuptools.build_meta\"\n\n[project]\nname = \"vlm-monitor\"\ndynamic = [\"version\"]\ndescription = \"A second set of eyes.\"\nreadme = \"README.md\"\nrequires-python = \">=3.10\"\nlicense = {text = \"Apache-2.0\"}\nauthors = [{name = \"Salman Naqvi\", email = \"s1148093@s.eduhk.hk\"}]\nkeywords = ['nbdev', 'vlms', 'llms', 'computer vision', 'natural language processing']\nclassifiers = [\n    \"Programming Language :: Python :: 3\",\n    \"Programming Language :: Python :: 3 :: Only\",\n]\ndependencies = ['fastcore','fastlite','fastllm','fastprogress','tiktoken','pycachy']\n\n[project.urls]\nRepository = \"https://github.com/ForBo7/vlm-monitor\"\nDocumentation = \"https://ForBo7.github.io/vlm-monitor/\"\n\n[project.entry-points.nbdev]\nvlm_monitor = \"vlm_monitor._modidx:d\"\n\n[tool.setuptools.dynamic]\nversion = {attr = \"vlm_monitor.__version__\"}\n\n[tool.setuptools.packages.find]\ninclude = [\"vlm_monitor\"]\n\n[tool.nbdev]\nallowed_metadata_keys = ['solveit']\nallowed_cell_metadata_keys = ['solveit_ai']"
}
```

</details>





找到了问题，就在 `pyproject.toml` 里：

```toml
dependencies = ['fastcore','fastlite','fastllm','fastprogress','tiktoken','pycachy']
```

`pip install fastllm` → 装的是 PyPI 上那个不知名的 `fastllm`，**不是**你的 `AnswerDotAI/fastllm`。那个 repo 没有 pip 包——安装方式应该是直接从 GitHub 拉。

你想怎么修？几个方向：

1. 依赖指向 GitHub repo：`dependencies = ['fastllm @ git+https://github.com/AnswerDotAI/fastllm']`
2. 把 fastllm 放到 CI 的 pip 安装步骤前单独装
3. 改名发布到 PyPI 然后正常声明依赖

你倾向哪个？🐛

<details class='token-usage-details' markdown='1'><summary>$0.0076</summary>

`total=156,338 | in=155,910 | out=428 | cached=98.8% | reasoning=194 | $0.0076 | deepseek-v4-flash`

</details>


give me the new line to copy paste into the toml file

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

把这一行：

```toml
dependencies = ['fastcore','fastlite','fastllm','fastprogress','tiktoken','pycachy']
```

改成：

```toml
dependencies = ['fastcore','fastlite','fastllm @ git+https://github.com/AnswerDotAI/fastllm','fastprogress','tiktoken','pycachy']
```

然后 `pip install -e '.[dev]'` 就会直接从 GitHub 拉下来了 🐛

<details class='token-usage-details' markdown='1'><summary>$0.0040</summary>

`total=78,613 | in=78,379 | out=234 | cached=98.3% | reasoning=130 | $0.0040 | deepseek-v4-flash`

</details>


i get this error now in my terminal?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

包名叫 `python-fastllm`，不是 `fastllm`。换个写法：

```toml
dependencies = ['fastcore','fastlite','python-fastllm @ git+https://github.com/AnswerDotAI/fastllm','fastprogress','tiktoken','pycachy']
```

这样依赖名和 GitHub 上的包名就一致了 🐛

<details class='token-usage-details' markdown='1'><summary>$0.0123</summary>

`total=78,895 | in=78,683 | out=212 | cached=70.3% | reasoning=131 | $0.0123 | deepseek-v4-flash`

</details>


## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()